# CineFusion — ALS (Matrix Factorization) + Hybrid Recommender

This notebook adds a **model-based collaborative filtering** layer using ALS (Alternating Least Squares),
which is the distributed equivalent of SVD. It then blends ALS scores with content-based scores
to produce a **Hybrid Recommender**.

### Pipeline Overview
```
ratings.csv  ──►  ALS Model  ──►  ALS Predictions
                                        │
tmdb embeddings  ──►  CBF Scores  ──────┼──►  Hybrid Score  ──►  Final Top-N
```

### Reads from Drive
- `ml-25m/ratings.csv`
- `ml-25m/movies.csv`
- `tmdb-5000/tmdb_5000_movies.csv`
- `tmdb-5000/tmdb_5000_credits.csv`
- `tmdb-5000-embeddings-BAAI/embeddings.npy`

### Saves to Drive
- `ml-25m-output/als_predictions_parquet/`
- `ml-25m-output/hybrid_scores_csv/`

## 0. Setup & Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/CineFusion"
print("Project folder found at:", BASE_DIR)

Mounted at /content/drive
Project folder found at: /content/drive/MyDrive/CineFusion


In [ ]:
import os

print(os.path.exists(BASE_DIR))   # should be True
print(os.listdir(BASE_DIR))       # should show ml-25m, tmdb-5000, etc.

True
['Project Proposal.gdoc', 'tmdb-5000', 'ml-100k', 'Copy_of_CineFusion.ipynb', 'Data Exploration.ipynb', 'Intermediate Project Report.gdoc', 'CineFusion.ipynb', 'ml-25m', 'ml-25m-output', 'tmdb-5000-embeddings-BAAI', 'CineFusion - CF.ipynb']


In [ ]:
# Install required libraries
!pip install -q pyspark sentence-transformers

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CineFusion - ALS") \
    .config("spark.driver.memory", "12g") \
    .config("spark.executor.memory", "12g") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.0.2


---
## PART 1 — ALS (Model-Based Collaborative Filtering)
ALS decomposes the user-movie rating matrix into latent user factors and movie factors.
This captures hidden taste dimensions (e.g. preference for action, drama, indie films).

### 1.1 Load & Explore Ratings

In [ ]:
from pyspark.sql import functions as F

ratings = spark.read.csv(
    f"{BASE_DIR}/ml-25m/ratings.csv",
    header=True,
    inferSchema=True
).select("userId", "movieId", "rating")

movies = spark.read.csv(
    f"{BASE_DIR}/ml-25m/movies.csv",
    header=True,
    inferSchema=True
).select("movieId", "title")

ratings = ratings.sample(fraction=0.2, seed=42)
print("Sampled ratings count:", ratings.count())

print("Ratings count:", ratings.count())
print("Movies count:", movies.count())

ratings.show(5)

Sampled ratings count: 5004496
Ratings count: 5004496
Movies count: 62423
+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     1|   1217|   3.5|
|     1|   2351|   4.5|
|     1|   2632|   5.0|
|     1|   4325|   5.0|
|     1|   5952|   4.0|
+------+-------+------+
only showing top 5 rows


In [ ]:
# Rating distribution — understand the data before modelling
ratings.groupBy("rating").count().orderBy("rating").show()

# Sparsity check
n_users = ratings.select("userId").distinct().count()
n_movies = ratings.select("movieId").distinct().count()
n_ratings = ratings.count()
sparsity = 1 - (n_ratings / (n_users * n_movies))

print(f"Users: {n_users}, Movies: {n_movies}")
print(f"Matrix sparsity: {sparsity:.4%}")
# Expected output: sparsity > 99% — this is why ALS handles it better than memory-based CF

+------+-------+
|rating|  count|
+------+-------+
|   0.5|  78967|
|   1.0| 155570|
|   1.5|  79818|
|   2.0| 328707|
|   2.5| 252502|
|   3.0| 980945|
|   3.5| 635072|
|   4.0|1329360|
|   4.5| 440513|
|   5.0| 723042|
+------+-------+

Users: 162375, Movies: 39606
Matrix sparsity: 99.9222%


### 1.2 Train/Test Split

In [ ]:
train, test = ratings.randomSplit([0.8, 0.2], seed=42)

print("Train size:", train.count())
print("Test size:", test.count())

Train size: 4003291
Test size: 1001205


### 1.3 Train ALS Model

Key hyperparameters:
- `rank` — number of latent factors (dimensionality of user/movie embeddings)
- `maxIter` — number of ALS iterations
- `regParam` — regularization to prevent overfitting
- `coldStartStrategy='drop'` — drops users/movies not seen in training (avoids NaN predictions)

In [ ]:
from pyspark.ml.recommendation import ALS

# als = ALS(
#     rank=100,
#     maxIter=15,
#     regParam=0.1,
#     userCol="userId",
#     itemCol="movieId",
#     ratingCol="rating",
#     coldStartStrategy="drop",
#     nonnegative=True,
#     seed=42
#)
als = ALS(
    rank=50,
    maxIter=10,
    regParam=0.1,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True,
    seed=42
)

print("Training ALS model...")
als_model = als.fit(train)
print("Training complete!")

Training ALS model...
Training complete!


### 1.4 Evaluate — RMSE & MAE

**Expected output:**
- RMSE around 0.80 – 0.90 (good range for MovieLens 25M)
- MAE around 0.60 – 0.70

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = als_model.transform(test)

evaluator_rmse = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

evaluator_mae = RegressionEvaluator(
    metricName="mae",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)

print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

RMSE: 0.8714
MAE:  0.6788


### 1.5 Generate Top-N ALS Recommendations Per User

**Expected output:** A DataFrame with columns `[userId, recommendations]`
where `recommendations` is a list of `(movieId, predicted_rating)` tuples — top 10 per user.

In [ ]:
# Generate top 10 movie recommendations for every user
user_recs = als_model.recommendForAllUsers(10)

user_recs.show(5, truncate=False)
print("Total users with recommendations:", user_recs.count())

+------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                                                                                                                                 |
+------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1     |[{102323, 5.29188}, {117531, 4.9196386}, {153024, 4.7485805}, {153014, 4.7485805}, {73194, 4.7485805}, {31524, 4.5880485}, {52413, 4.574662}, {34231, 4.5707445}, {49183, 4.537211}, {175169, 4.5317483}]       |
|3     |[{102323, 5.555064}, {117531, 5.4392605}, {175169, 5.3632584}, {128187, 5.334512}, {184299, 5.266841}, {192289, 5.216628

In [ ]:
# Flatten recommendations into (userId, movieId, als_score) rows
from pyspark.sql.functions import explode, col

als_flat = user_recs.select(
    col("userId"),
    explode("recommendations").alias("rec")
).select(
    col("userId"),
    col("rec.movieId").alias("movieId"),
    col("rec.rating").alias("als_score")
)

# Join with movie titles
als_flat = als_flat.join(movies, "movieId")

als_flat.show(10, truncate=False)

# Expected output: rows like:
# userId=1, movieId=318, title="Shawshank Redemption", als_score=4.87

+-------+------+---------+------------------------------------------------------------------------------------+
|movieId|userId|als_score|title                                                                               |
+-------+------+---------+------------------------------------------------------------------------------------+
|102323 |1     |5.29188  |Grin Without a Cat, A (Fond de l'air est rouge, Le) (1977)                          |
|117531 |1     |4.9196386|Watermark (2014)                                                                    |
|153024 |1     |4.7485805|Voices Through Time (1996)                                                          |
|153014 |1     |4.7485805|Nightfall (1999)                                                                    |
|73194  |1     |4.7485805|Whole Night, A (Toute une nuit) (1982)                                              |
|31524  |1     |4.5880485|Bitter Tears of Petra von Kant, The (bitteren Tränen der Petra von Kant, Die) 

### 1.6 Inspect Latent Factors

ALS learns a 100-dimensional embedding for each user and movie.
These are the "taste dimensions" learned from rating patterns.

In [ ]:
user_factors = als_model.userFactors
item_factors = als_model.itemFactors

print("User factors shape: ", user_factors.count(), "users x", len(user_factors.first()["features"]), "dims")
print("Item factors shape: ", item_factors.count(), "movies x", len(item_factors.first()["features"]), "dims")

# Sample: first user's latent vector
print("\nSample user latent vector (first 10 dims):")
print(user_factors.first()["features"][:10])

User factors shape:  161990 users x 50 dims
Item factors shape:  36970 movies x 50 dims

Sample user latent vector (first 10 dims):
[0.11871211230754852, 0.11568226665258408, 0.47187522053718567, 0.28722941875457764, 0.14077123999595642, 0.20045249164104462, 0.3036686182022095, 0.0, 1.0044862031936646, 0.03121517412364483]


### 1.7 Save ALS Predictions to Drive

In [ ]:
from pyspark import StorageLevel

als_flat = als_flat.persist(StorageLevel.MEMORY_AND_DISK)

als_flat.coalesce(1).write \
    .mode("overwrite") \
    .parquet(f"{BASE_DIR}/ml-25m-output/als_predictions_parquet")

print(f"Saved {als_flat.count()} ALS predictions to Drive")

Saved 1619900 ALS predictions to Drive


---
## PART 2 — Hybrid Recommender
Blend ALS (collaborative) scores with content-based (BGE embedding) scores
into a single unified recommendation score.

Formula: `hybrid_score = α * als_score_norm + (1 - α) * cbf_score`

We use `α = 0.6` (favor collaborative signal slightly, tunable).

### 2.1 Load Content-Based Data (TMDB + BGE Embeddings)

In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.metrics.pairwise import cosine_similarity

tmdb_movies = pd.read_csv(f"{BASE_DIR}/tmdb-5000/tmdb_5000_movies.csv")
credits = pd.read_csv(f"{BASE_DIR}/tmdb-5000/tmdb_5000_credits.csv")
tmdb_movies = tmdb_movies.merge(credits, on="title")

# Recreate tags (same as original notebook)
def extract_names(obj):
    try:
        obj = ast.literal_eval(obj)
        return [i['name'] for i in obj]
    except:
        return []

def extract_top_cast(obj):
    try:
        obj = ast.literal_eval(obj)
        return [i['name'] for i in obj[:3]]
    except:
        return []

def extract_director(obj):
    try:
        obj = ast.literal_eval(obj)
        for i in obj:
            if i['job'] == 'Director':
                return [i['name']]
        return []
    except:
        return []

tmdb_movies["genres"] = tmdb_movies["genres"].apply(extract_names)
tmdb_movies["keywords"] = tmdb_movies["keywords"].apply(extract_names)
tmdb_movies["cast"] = tmdb_movies["cast"].apply(extract_top_cast)
tmdb_movies["director"] = tmdb_movies["crew"].apply(extract_director)
tmdb_movies["tags"] = tmdb_movies.apply(
    lambda x: " ".join(x["genres"] + x["keywords"] + x["cast"] + x["director"] + [str(x["overview"])]),
    axis=1
)

tmdb_movies = tmdb_movies[["title", "tags"]].reset_index(drop=True)

# Load pre-computed BGE embeddings
embeddings = np.load(f"{BASE_DIR}/tmdb-5000-embeddings-BAAI/embeddings.npy")

print("TMDB movies:", len(tmdb_movies))
print("Embeddings shape:", embeddings.shape)

TMDB movies: 4809
Embeddings shape: (4809, 768)


### 2.2 Content-Based Score Function

For a given seed movie, returns similarity scores against all TMDB movies using BGE embeddings.

In [ ]:
def get_cbf_scores(seed_movie_title, top_n=10):
    """
    Returns top-N similar movies using BGE embedding cosine similarity.
    Output: list of (title, cbf_score) sorted descending.
    """
    matches = tmdb_movies[tmdb_movies["title"] == seed_movie_title]
    if matches.empty:
        print(f"Movie '{seed_movie_title}' not found in TMDB dataset.")
        return []

    idx = matches.index[0]
    scores = cosine_similarity([embeddings[idx]], embeddings)[0]
    top_indices = scores.argsort()[::-1][1:top_n+1]

    return [(tmdb_movies.iloc[i].title, float(scores[i])) for i in top_indices]

# Quick test
print("CBF results for Avatar:")
for title, score in get_cbf_scores("Avatar"):
    print(f"  {title}: {score:.4f}")

CBF results for Avatar:
  Battle: Los Angeles: 0.7222
  Terminator Genisys: 0.7128
  Starship Troopers: 0.7091
  Æon Flux: 0.7031
  Prometheus: 0.7021
  Supernova: 0.6984
  The Inhabited Island: 0.6962
  Aliens: 0.6959
  Damnation Alley: 0.6959
  Battlefield Earth: 0.6936


### 2.3 Hybrid Recommendation Function

Given a `userId` and a seed movie they liked:
1. Get ALS top-N predictions for that user
2. Get CBF top-N similar movies to the seed
3. Normalize both scores to [0, 1]
4. Blend with `alpha` weight and return ranked results

**Expected output:** A ranked DataFrame with columns `[title, als_score_norm, cbf_score, hybrid_score]`

In [ ]:
def normalize(series):
    """Min-max normalize a pandas Series to [0, 1]."""
    min_val, max_val = series.min(), series.max()
    if max_val == min_val:
        return series * 0
    return (series - min_val) / (max_val - min_val)


def hybrid_recommend(user_id, seed_movie_title, alpha=0.6, top_n=10):
    """
    Hybrid recommendation blending ALS + CBF.

    Parameters:
        user_id         : int  — the target user
        seed_movie_title: str  — a movie the user liked (used for CBF signal)
        alpha           : float — weight for ALS score (1-alpha goes to CBF)
        top_n           : int  — number of recommendations to return

    Returns:
        pandas DataFrame with hybrid ranked recommendations
    """

    # --- ALS side ---
    als_user = als_flat.filter(col("userId") == user_id).toPandas()
    if als_user.empty:
        print(f"No ALS predictions found for user {user_id}.")
        return None

    als_user["als_score_norm"] = normalize(als_user["als_score"])
    als_titles = set(als_user["title"].tolist())

    # --- CBF side ---
    cbf_results = get_cbf_scores(seed_movie_title, top_n=50)
    if not cbf_results:
        return None

    cbf_df = pd.DataFrame(cbf_results, columns=["title", "cbf_score"])
    cbf_df["cbf_score"] = normalize(cbf_df["cbf_score"])

    # --- Merge on title ---
    merged = pd.merge(als_user[["title", "als_score_norm"]], cbf_df, on="title", how="outer").fillna(0)

    # --- Blend ---
    merged["hybrid_score"] = alpha * merged["als_score_norm"] + (1 - alpha) * merged["cbf_score"]
    merged = merged.sort_values("hybrid_score", ascending=False).head(top_n)

    return merged.reset_index(drop=True)


print("Hybrid recommender function ready.")

Hybrid recommender function ready.


### 2.4 Run Hybrid Recommendations — Sample Users

**Expected output:** A clean table showing titles ranked by hybrid score,
with both ALS and CBF sub-scores visible for interpretability.

In [ ]:
# Test with a sample user
sample_user_id = 1
seed_movie = "Avatar"

results = hybrid_recommend(
    user_id=sample_user_id,
    seed_movie_title=seed_movie,
    alpha=0.6,
    top_n=10
)

print(f"\nHybrid Recommendations for User {sample_user_id} (seed: '{seed_movie}')")
print("=" * 65)
print(results.to_string(index=False))

# Expected output example:
#  title                          als_score_norm  cbf_score  hybrid_score
#  Guardians of the Galaxy              0.95       0.72          0.86
#  Interstellar                         0.88       0.81          0.85
#  ...


Hybrid Recommendations for User 1 (seed: 'Avatar')
                                                     title  als_score_norm  cbf_score  hybrid_score
Grin Without a Cat, A (Fond de l'air est rouge, Le) (1977)        1.000000   0.000000      0.600000
                                       Battle: Los Angeles        0.000000   1.000000      0.400000
                                        Terminator Genisys        0.000000   0.817181      0.326872
                                          Watermark (2014)        0.510293   0.000000      0.306176
                                         Starship Troopers        0.000000   0.745765      0.298306
                                                  Æon Flux        0.000000   0.628297      0.251319
                                                Prometheus        0.000000   0.609489      0.243796
                                                 Supernova        0.000000   0.538032      0.215213
                                      The Inhabi

### 2.5 Compare: ALS Only vs CBF Only vs Hybrid

Shows the value of blending — each method catches different movies.

In [ ]:
sample_user_id = 1
seed_movie = "Avatar"

# ALS only
als_only = als_flat.filter(col("userId") == sample_user_id).toPandas()
als_only = als_only.sort_values("als_score", ascending=False).head(10)

# CBF only
cbf_only = get_cbf_scores(seed_movie, top_n=10)
cbf_only_titles = [t for t, _ in cbf_only]

# Hybrid
hybrid_results = hybrid_recommend(sample_user_id, seed_movie)
hybrid_titles = hybrid_results["title"].tolist() if hybrid_results is not None else []

print("ALS Only Top 10:")
for t in als_only["title"].tolist():
    print(f"  {'✅' if t in hybrid_titles else '  '} {t}")

print("\nCBF Only Top 10:")
for t in cbf_only_titles:
    print(f"  {'✅' if t in hybrid_titles else '  '} {t}")

print("\nHybrid Top 10:")
for t in hybrid_titles:
    print(f"  {t}")

# ✅ marks movies that made it into the hybrid list

ALS Only Top 10:
  ✅ Grin Without a Cat, A (Fond de l'air est rouge, Le) (1977)
  ✅ Watermark (2014)
     Voices Through Time (1996)
     Nightfall (1999)
     Whole Night, A (Toute une nuit) (1982)
     Bitter Tears of Petra von Kant, The (bitteren Tränen der Petra von Kant, Die) (1972)
     Ulysses' Gaze (To Vlemma tou Odyssea) (1995)
     Fighter in the Wind (2004)
     Vital (2004)
     Rumble: The Indians Who Rocked the World (2017)

CBF Only Top 10:
  ✅ Battle: Los Angeles
  ✅ Terminator Genisys
  ✅ Starship Troopers
  ✅ Æon Flux
  ✅ Prometheus
  ✅ Supernova
  ✅ The Inhabited Island
  ✅ Aliens
     Damnation Alley
     Battlefield Earth

Hybrid Top 10:
  Grin Without a Cat, A (Fond de l'air est rouge, Le) (1977)
  Battle: Los Angeles
  Terminator Genisys
  Watermark (2014)
  Starship Troopers
  Æon Flux
  Prometheus
  Supernova
  The Inhabited Island
  Aliens


### 2.6 Alpha Sensitivity Analysis

See how changing the blend weight shifts recommendations.
Useful for tuning — higher alpha = trust collaborative more, lower = trust content more.

In [ ]:
sample_user_id = 1
seed_movie = "Avatar"

for alpha in [0.2, 0.5, 0.8]:
    result = hybrid_recommend(sample_user_id, seed_movie, alpha=alpha, top_n=5)
    if result is not None:
        print(f"\n--- Alpha = {alpha} (ALS weight={alpha}, CBF weight={1-alpha}) ---")
        print(result[["title", "hybrid_score"]].to_string(index=False))


--- Alpha = 0.2 (ALS weight=0.2, CBF weight=0.8) ---
              title  hybrid_score
Battle: Los Angeles      0.800000
 Terminator Genisys      0.653745
  Starship Troopers      0.596612
           Æon Flux      0.502638
         Prometheus      0.487591

--- Alpha = 0.5 (ALS weight=0.5, CBF weight=0.5) ---
                                                     title  hybrid_score
                                       Battle: Los Angeles      0.500000
Grin Without a Cat, A (Fond de l'air est rouge, Le) (1977)      0.500000
                                        Terminator Genisys      0.408590
                                         Starship Troopers      0.372882
                                                  Æon Flux      0.314149

--- Alpha = 0.8 (ALS weight=0.8, CBF weight=0.19999999999999996) ---
                                                     title  hybrid_score
Grin Without a Cat, A (Fond de l'air est rouge, Le) (1977)      0.800000
                                  

### 2.7 Save Hybrid Scores to Drive

In [ ]:
# Generate hybrid results for a sample of users and save
sample_users = [1, 2, 3, 5, 10]  # extend as needed
seed_movie = "Avatar"

all_results = []
for uid in sample_users:
    res = hybrid_recommend(uid, seed_movie, alpha=0.6, top_n=10)
    if res is not None:
        res["userId"] = uid
        all_results.append(res)

if all_results:
    import pandas as pd
    combined = pd.concat(all_results, ignore_index=True)

    out_path = f"{BASE_DIR}/ml-25m-output/hybrid_scores_csv"
    os.makedirs(out_path, exist_ok=True)
    combined.to_csv(f"{out_path}/hybrid_recommendations.csv", index=False)

    print(f"Saved hybrid recommendations for {len(sample_users)} users to Drive")
    print(combined.head(10).to_string(index=False))

Saved hybrid recommendations for 5 users to Drive
                                                     title  als_score_norm  cbf_score  hybrid_score  userId
Grin Without a Cat, A (Fond de l'air est rouge, Le) (1977)        1.000000   0.000000      0.600000       1
                                       Battle: Los Angeles        0.000000   1.000000      0.400000       1
                                        Terminator Genisys        0.000000   0.817181      0.326872       1
                                          Watermark (2014)        0.510293   0.000000      0.306176       1
                                         Starship Troopers        0.000000   0.745765      0.298306       1
                                                  Æon Flux        0.000000   0.628297      0.251319       1
                                                Prometheus        0.000000   0.609489      0.243796       1
                                                 Supernova        0.000000   0.538032 

---
## PART 3 — Summary & Results

| Metric | Value |
|---|---|
| ALS RMSE (test set) | ~0.80–0.90 |
| ALS MAE (test set) | ~0.60–0.70 |
| Latent factors (rank) | 100 |
| Users with recommendations | ~162,000 |
| Hybrid alpha used | 0.6 |

### What this notebook contributes to CineFusion
- **ALS** adds model-based CF on top of the existing memory-based CF (MinHash + Pearson)
- **Hybrid blending** unifies all three signals: ALS + TF-IDF/BGE content-based
- Saved outputs (`als_predictions_parquet`, `hybrid_scores_csv`) can be consumed by a future serving layer or evaluation notebook